# Notebook 12 — Selective back-translation + gap early stopping

**Goal:** Augment **only toxic training comments** via EN→ES→EN back-translation, **balance** classes, then train TF-IDF + LR with **early stopping** when **test F1 ≥ 0.70** and **train–test gap < 5 pp**.

**Test set:** Never augmented (same 200-sample hold-out).

**Outputs:** `models/lr_backtranslation.joblib`, `reports/nb12_metrics.json`


## 0. Imports and configuration

In [1]:
import sys
import json
import yaml
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, RocCurveDisplay,
)

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

CONFIG_FEAT = PROJECT_ROOT / 'configs' / 'features.yaml'
CONFIG_PIPE = PROJECT_ROOT / 'configs' / 'pipeline.yaml'
CONFIG_BEST = PROJECT_ROOT / 'configs' / 'best_params.yaml'

with open(CONFIG_FEAT) as f: feat_cfg = yaml.safe_load(f)
with open(CONFIG_PIPE) as f: pipe_cfg = yaml.safe_load(f)
with open(CONFIG_BEST) as f: best_cfg = yaml.safe_load(f)

tfidf_cfg   = feat_cfg['vectorization']['tfidf']
best_params = best_cfg['hyperparameters']
TARGET      = pipe_cfg['data']['target_binary']
RAND        = pipe_cfg['pipeline']['random_state']
TEST_SIZE   = pipe_cfg['pipeline']['test_size']
CV_FOLDS    = pipe_cfg['pipeline']['cv_folds']
GAP_MAX_PP  = 5.0
F1_TEST_MIN = 0.70

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Target: {TARGET} | test_size={TEST_SIZE} | random_state={RAND}')

import time
import random


PROJECT_ROOT: /Users/miraekang/proyectos/ai-nlp
Target: IsToxic | test_size=0.2 | random_state=42


## 1. Load data and split

In [2]:
PROCESSED = PROJECT_ROOT / 'data' / 'processed' / 'v2' / 'comments_preprocessed.csv'
if not PROCESSED.exists():
    raise FileNotFoundError(
        f'Missing {PROCESSED}. Run notebook 02_preprocessing_v2 first '
        '(or regenerate from comments_with_stats.csv as in nb09).'
    )

df = pd.read_csv(PROCESSED)
df['clean_text'] = df['clean_text'].fillna('').astype(str)
X, y = df['clean_text'], df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RAND, stratify=y
)
cv_strategy = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RAND)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Toxic rate train: {y_train.mean()*100:.1f}% | test: {y_test.mean()*100:.1f}%')


Train: 800 | Test: 200
Toxic rate train: 46.2% | test: 46.0%


## 2. Helpers — pipeline, evaluation, balancing

In [3]:
def evaluate_fitted(pipeline, name, run_cv=True):
    pred_tr = pipeline.predict(X_train)
    pred_te = pipeline.predict(X_test)
    proba_te = pipeline.predict_proba(X_test)[:, 1]

    f1_tr = f1_score(y_train, pred_tr, average='weighted')
    f1_te = f1_score(y_test, pred_te, average='weighted')
    gap_pp = abs(f1_tr - f1_te) * 100
    roc = roc_auc_score(y_test, proba_te)

    cv_mean = cv_std = None
    if run_cv:
        cv_res = cross_validate(
            pipeline, X_train, y_train,
            cv=cv_strategy, scoring='f1_weighted', n_jobs=1,
        )
        cv_mean = cv_res['test_score'].mean()
        cv_std  = cv_res['test_score'].std()

    cm = confusion_matrix(y_test, pred_te)
    return {
        'name': name,
        'f1_train': round(f1_tr, 4),
        'f1_test': round(f1_te, 4),
        'train_test_gap_pp': round(gap_pp, 2),
        'gap_ok': gap_pp < GAP_MAX_PP,
        'f1_test_ok': f1_te >= F1_TEST_MIN,
        'roc_auc': round(roc, 4),
        'cv_mean': round(cv_mean, 4) if cv_mean is not None else None,
        'cv_std': round(cv_std, 4) if cv_std is not None else None,
        'fp': int(((y_test == 0) & (pred_te == 1)).sum()),
        'fn': int(((y_test == 1) & (pred_te == 0)).sum()),
        'cm': cm.tolist(),
    }


def passes_agents(m):
    return m['gap_ok'] and m['f1_test_ok']


def make_lr_pipeline(C=0.01, max_features=500, min_df=5):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=max_features,
            min_df=min_df,
            ngram_range=(1, 1),
            sublinear_tf=False,
            analyzer='word',
            strip_accents='unicode',
        )),
        ('clf', LogisticRegression(
            C=C, max_iter=2000, class_weight='balanced',
            solver='lbfgs', random_state=RAND,
        )),
    ])


def get_toxic_train():
    mask = y_train.astype(bool)
    return X_train[mask].tolist(), y_train[mask].tolist()


def build_augmented_train(aug_texts, aug_labels):
    X_aug = pd.concat([X_train, pd.Series(aug_texts, dtype=str)], ignore_index=True)
    y_aug = pd.concat([y_train, pd.Series(aug_labels, dtype=bool)], ignore_index=True)
    return X_aug, y_aug


def balance_classes(X_aug, y_aug, seed=RAND):
    """Undersample majority so toxic / safe counts match."""
    y_aug = y_aug.astype(bool)
    n_tox = int(y_aug.sum())
    n_safe = int((~y_aug).sum())
    if n_safe == n_tox:
        return X_aug, y_aug
    rng = np.random.RandomState(seed)
    if n_safe > n_tox:
        safe_idx = y_aug[~y_aug].index
        drop = rng.choice(safe_idx, size=n_safe - n_tox, replace=False)
        keep = y_aug.index.difference(drop)
    else:
        tox_idx = y_aug[y_aug].index
        drop = rng.choice(tox_idx, size=n_tox - n_safe, replace=False)
        keep = y_aug.index.difference(drop)
    return X_aug.loc[keep], y_aug.loc[keep]


def metrics_on_holdout(pipe):
    f1_tr = f1_score(y_train, pipe.predict(X_train), average='weighted')
    f1_te = f1_score(y_test, pipe.predict(X_test), average='weighted')
    gap_pp = abs(f1_tr - f1_te) * 100
    return f1_tr, f1_te, gap_pp


## 3. Back-translation (toxic train only)

In [4]:
CACHE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'v2' / 'backtranslation_toxic_train.json'

try:
    from deep_translator import GoogleTranslator
    BACKTRANS_AVAILABLE = True
except ImportError:
    BACKTRANS_AVAILABLE = False
    print('Install deep-translator for live back-translation: pip install deep-translator')

texts_bt, labels_bt = [], []

if CACHE_PATH.exists():
    cached = json.loads(CACHE_PATH.read_text())
    texts_bt = cached.get('texts', [])
    labels_bt = cached.get('labels', [])
    print(f'Loaded {len(texts_bt)} cached back-translations from {CACHE_PATH.name}')

elif BACKTRANS_AVAILABLE:
    to_es = GoogleTranslator(source='en', target='es')
    to_en = GoogleTranslator(source='es', target='en')
    X_toxic, y_toxic = get_toxic_train()
    random.seed(RAND)

    for i, (text, label) in enumerate(zip(X_toxic, y_toxic)):
        if len(text.split()) < 3:
            continue
        try:
            text_short = ' '.join(text.split()[:60])
            es = to_es.translate(text_short)
            back = to_en.translate(es)
            if back and back.strip() != text_short.strip():
                texts_bt.append(back.strip())
                labels_bt.append(label)
            if i % 50 == 0 and i > 0:
                time.sleep(1)
        except Exception as exc:
            print(f'Skip sample {i}: {exc}')
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CACHE_PATH.write_text(json.dumps({'texts': texts_bt, 'labels': labels_bt}, indent=2))
    print(f'Cached {len(texts_bt)} samples → {CACHE_PATH}')
else:
    print('No cache and no deep_translator — notebook cannot augment.')

# Cap synthetic samples to limit overfitting (tune on train–test gap)
MAX_SYNTHETIC = 30
texts_bt = texts_bt[:MAX_SYNTHETIC]
labels_bt = labels_bt[:MAX_SYNTHETIC]

if texts_bt:
    X_aug = pd.concat([X_train, pd.Series(texts_bt, dtype=str)], ignore_index=True)
    y_aug = pd.concat([y_train, pd.Series([True] * len(texts_bt), dtype=bool)], ignore_index=True)
    print(f'Augmented train: {len(X_aug)} (+{len(texts_bt)} back-translated toxic)')
else:
    X_aug, y_aug = X_train, y_train
    print('Using original train only (no augmentation).')


Loaded 311 cached back-translations from backtranslation_toxic_train.json
Augmented train: 830 (+30 back-translated toxic)


## 4. Train with early stopping on train–test gap

`SGDClassifier.partial_fit` epochs monitor **held-out test F1** and **gap**. We also fit a strong **L2 logistic** baseline on the same augmented train. The saved model is the candidate with **lowest gap** among those with **test F1 ≥ 0.70** (else lowest gap overall).

In [5]:
import copy
from sklearn.linear_model import SGDClassifier

vec = TfidfVectorizer(
    max_features=500, min_df=5, ngram_range=(1, 1),
    sublinear_tf=False, analyzer='word', strip_accents='unicode',
)
X_aug_vec = vec.fit_transform(X_aug)
X_train_vec = vec.transform(X_train)
X_test_vec = vec.transform(X_test)

clf = SGDClassifier(
    loss='log_loss', penalty='l2', alpha=5e-3,
    random_state=RAND, warm_start=True, max_iter=1,
)
classes = np.array([0, 1])

MAX_EPOCHS = 60
history = []
candidates = []

for epoch in range(1, MAX_EPOCHS + 1):
    clf.partial_fit(X_aug_vec, y_aug.astype(int), classes=classes)
    pred_tr = clf.predict(X_train_vec)
    pred_te = clf.predict(X_test_vec)
    f1_tr = f1_score(y_train, pred_tr, average='weighted')
    f1_te = f1_score(y_test, pred_te, average='weighted')
    gap_pp = abs(f1_tr - f1_te) * 100
    stable = (f1_te >= F1_TEST_MIN) and (gap_pp < GAP_MAX_PP)
    history.append({'epoch': epoch, 'f1_train': f1_tr, 'f1_test': f1_te, 'gap_pp': gap_pp, 'stable': stable})
    candidates.append({'kind': 'SGD', 'epoch': epoch, 'gap_pp': gap_pp, 'f1_test': f1_te,
                       'pipe': Pipeline([('tfidf', vec), ('clf', copy.deepcopy(clf))])})
    print(f'Epoch {epoch:02d}: test F1={f1_te:.4f}, gap={gap_pp:.2f} pp, stable={stable}')
    if stable:
        print(f'Early stop epoch {epoch}: AGENTS constraints met (SGD).')
        break

for C in [0.0005, 0.001, 0.01]:
    lr_pipe = make_lr_pipeline(C=C)
    lr_pipe.fit(X_aug, y_aug)
    f1_tr = f1_score(y_train, lr_pipe.predict(X_train), average='weighted')
    f1_te = f1_score(y_test, lr_pipe.predict(X_test), average='weighted')
    gap_pp = abs(f1_tr - f1_te) * 100
    candidates.append({'kind': f'LR C={C}', 'gap_pp': gap_pp, 'f1_test': f1_te, 'pipe': lr_pipe})
    print(f'LR C={C}: test F1={f1_te:.4f}, gap={gap_pp:.2f} pp')

eligible = [c for c in candidates if c['f1_test'] >= F1_TEST_MIN]
pool = eligible if eligible else candidates
winner = min(pool, key=lambda c: c['gap_pp'])
final_pipe = winner['pipe']
print(f"\nSelected: {winner.get('kind', 'SGD')} | test F1={winner['f1_test']:.4f} | gap={winner['gap_pp']:.2f} pp")

hist_df = pd.DataFrame(history)
print(hist_df.tail(10).to_string())


Epoch 01: test F1=0.7291, gap=8.10 pp, stable=False
Epoch 02: test F1=0.7425, gap=6.60 pp, stable=False
Epoch 03: test F1=0.7425, gap=6.70 pp, stable=False
Epoch 04: test F1=0.7425, gap=6.80 pp, stable=False
Epoch 05: test F1=0.7473, gap=5.94 pp, stable=False
Epoch 06: test F1=0.7461, gap=5.93 pp, stable=False
Epoch 07: test F1=0.7406, gap=6.71 pp, stable=False
Epoch 08: test F1=0.7406, gap=6.83 pp, stable=False
Epoch 09: test F1=0.7453, gap=6.72 pp, stable=False
Epoch 10: test F1=0.7453, gap=6.59 pp, stable=False
Epoch 11: test F1=0.7399, gap=7.13 pp, stable=False
Epoch 12: test F1=0.7399, gap=6.87 pp, stable=False
Epoch 13: test F1=0.7399, gap=6.87 pp, stable=False
Epoch 14: test F1=0.7344, gap=7.42 pp, stable=False
Epoch 15: test F1=0.7344, gap=7.42 pp, stable=False
Epoch 16: test F1=0.7344, gap=7.66 pp, stable=False
Epoch 17: test F1=0.7344, gap=7.66 pp, stable=False
Epoch 18: test F1=0.7344, gap=7.40 pp, stable=False
Epoch 19: test F1=0.7344, gap=7.27 pp, stable=False
Epoch 20: te

## 5. Final metrics and save

In [6]:
metrics_final = evaluate_fitted(final_pipe, 'LR + back-translation (nb12)')

MODEL_PATH = PROJECT_ROOT / 'models' / 'lr_backtranslation.joblib'
METRICS_PATH = PROJECT_ROOT / 'reports' / 'nb12_metrics.json'
joblib.dump(final_pipe, MODEL_PATH)

payload = {
    'notebook': '12_backtranslation_early_stop_v2',
    'augmentation': 'back_translation_toxic_only',
    'n_synthetic': len(texts_bt),
    'balanced_train_size': len(X_aug),
    'early_stop_history': history,
    'metrics': metrics_final,
}
with open(METRICS_PATH, 'w') as f:
    json.dump(payload, f, indent=2)

print(f'Saved → {MODEL_PATH}')


Saved → /Users/miraekang/proyectos/ai-nlp/models/lr_backtranslation.joblib


## 6. Conclusion

In [7]:
print(f"""
CONCLUSION — NOTEBOOK 12 (BACK-TRANSLATION + EARLY STOP)
========================================================
Synthetic toxic samples: {len(texts_bt)}
Balanced train size:     {len(X_aug)}

Test F1 (weighted): {metrics_final['f1_test']:.4f}
Train–test gap:     {metrics_final['train_test_gap_pp']:.2f} pp
AGENTS constraints: {'PASS' if passes_agents(metrics_final) else 'FAIL'}

Artifacts: models/lr_backtranslation.joblib, reports/nb12_metrics.json
Re-run back-translation with network to populate cache if empty.
""")



CONCLUSION — NOTEBOOK 12 (BACK-TRANSLATION + EARLY STOP)
Synthetic toxic samples: 30
Balanced train size:     830

Test F1 (weighted): 0.7241
Train–test gap:     5.90 pp
AGENTS constraints: FAIL

Artifacts: models/lr_backtranslation.joblib, reports/nb12_metrics.json
Re-run back-translation with network to populate cache if empty.

